# Notebook 06: Key Expiry, TTL & Key Management

One of Redis's most powerful features is **automatic key expiration**. You can set a "time to live" (TTL) on any key, and Redis will **automatically delete it** when the time is up.

This is the secret sauce behind:
- **Caching** — cached data auto-expires so it stays fresh
- **Sessions** — login sessions auto-expire for security
- **Rate limiting** — counters auto-reset
- **OTP/Tokens** — verification codes that self-destruct

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## 1. Setting Expiry with SET

In [ ]:
# EX = seconds
# Redis CLI: SET cache:data "value" EX 30
r.set('cache:data', 'some cached value', ex=30)
print(f"cache:data TTL: {r.ttl('cache:data')} seconds")

# PX = milliseconds
# Redis CLI: SET flash:msg "Sale!" PX 5000
r.set('flash:msg', 'Flash sale!', px=5000)
print(f"flash:msg TTL: {r.pttl('flash:msg')} milliseconds")

## 2. EXPIRE / PEXPIRE — Set Expiry on Existing Keys

In [ ]:
# Create a key without expiry
r.set('mykey', 'hello')
print(f"TTL before EXPIRE: {r.ttl('mykey')}")  # -1 means no expiry

# EXPIRE — Set expiry in seconds
# Redis CLI: EXPIRE mykey 60
r.expire('mykey', 60)
print(f"TTL after EXPIRE 60: {r.ttl('mykey')} seconds")

# PEXPIRE — Set expiry in milliseconds
# Redis CLI: PEXPIRE mykey 30000
r.pexpire('mykey', 30000)
print(f"PTTL after PEXPIRE 30000: {r.pttl('mykey')} ms")

## 3. EXPIREAT — Expire at a Specific Time

In [ ]:
import datetime

r.set('event:sale', 'Summer Sale 50% Off')

# Expire at a specific Unix timestamp (10 seconds from now)
expire_at = int(time.time()) + 10
r.expireat('event:sale', expire_at)

expire_time = datetime.datetime.fromtimestamp(expire_at)
print(f"Key expires at: {expire_time}")
print(f"TTL: {r.ttl('event:sale')} seconds")

## 4. TTL / PTTL — Check Remaining Time

Return values:
- **Positive number** = remaining time
- **-1** = key exists but has NO expiry (permanent)
- **-2** = key does NOT exist

In [ ]:
# Key with expiry
r.set('temp', 'value', ex=100)
print(f"Key with expiry   → TTL: {r.ttl('temp')}")

# Key without expiry
r.set('permanent', 'value')
print(f"Permanent key     → TTL: {r.ttl('permanent')} (-1 = no expiry)")

# Non-existent key
print(f"Non-existent key  → TTL: {r.ttl('ghost')} (-2 = doesn't exist)")

# PTTL for millisecond precision
print(f"Millisecond TTL   → PTTL: {r.pttl('temp')} ms")

## 5. PERSIST — Remove Expiry (Make Permanent Again)

In [ ]:
r.set('important', 'data', ex=30)
print(f"Before PERSIST — TTL: {r.ttl('important')}")

# Redis CLI: PERSIST important
r.persist('important')
print(f"After PERSIST  — TTL: {r.ttl('important')} (-1 = permanent!)")

## 6. Watch a Key Expire in Real-Time

In [ ]:
r.set('countdown', 'I will disappear!', ex=5)

print("Watching a key expire:")
for i in range(7):
    value = r.get('countdown')
    ttl = r.ttl('countdown')
    if value:
        print(f"  t={i}s: value='{value}', TTL={ttl}s")
    else:
        print(f"  t={i}s: KEY EXPIRED! (TTL={ttl})")
    time.sleep(1)

print("\nThe key automatically deleted itself!")

---
## 7. How Expiry Works Internally

Redis uses two strategies to clean up expired keys:

1. **Lazy Deletion** — When you try to access a key, Redis checks if it's expired. If yes, it deletes it and returns nil.

2. **Active Deletion** — 10 times per second, Redis randomly samples 20 keys with expiry. It deletes any that are expired. If > 25% were expired, it repeats immediately.

This means expired keys don't consume memory forever, even if nobody accesses them.

---
## 8. Key Naming Conventions

Good key naming makes your Redis data organized and searchable.

| Pattern | Example | Use Case |
|---|---|---|
| `object:id:field` | `user:1001:email` | Specific field |
| `object:id` | `user:1001` | Hash containing all fields |
| `action:scope` | `cache:homepage` | Cached data |
| `scope:date` | `visits:2024-01-15` | Time-scoped data |
| `scope:id:action` | `ratelimit:user1:api` | Rate limiting |

### Rules of Thumb
- Use **colons** (`:`) as separators
- Keep keys **short but descriptive**
- Use **consistent prefixes** for easy scanning
- Avoid very long keys (wastes memory)

---
## 9. KEYS vs SCAN — Finding Keys Safely

In [ ]:
# Create some test data
for i in range(20):
    r.set(f'user:{i}:name', f'User_{i}')
    r.set(f'product:{i}:name', f'Product_{i}')

# KEYS — Simple but DANGEROUS in production (blocks server!)
# Redis CLI: KEYS user:*
user_keys = r.keys('user:*')
print(f"KEYS found {len(user_keys)} user keys")
print(f"First 5: {sorted(user_keys)[:5]}")

print("\n⚠️  KEYS blocks the server while scanning ALL keys!")
print("   Never use in production with large databases!")

In [ ]:
# SCAN — Safe, cursor-based iteration (use this in production!)
# Redis CLI: SCAN 0 MATCH user:* COUNT 10

print("SCAN results (safe iteration):")
count = 0
for key in r.scan_iter('user:*', count=10):
    count += 1
    if count <= 5:
        print(f"  {key}")

print(f"  ... total: {count} keys found")
print("\nSCAN doesn't block — it yields results incrementally!")

## 10. TYPE, RENAME, and Other Key Commands

In [ ]:
r.set('mystring', 'hello')
r.lpush('mylist', 'a', 'b', 'c')
r.sadd('myset', 'x', 'y', 'z')
r.hset('myhash', 'field', 'value')

# TYPE — What kind of data structure is stored?
# Redis CLI: TYPE mystring
print(f"mystring type: {r.type('mystring')}")  # string
print(f"mylist type:   {r.type('mylist')}")     # list
print(f"myset type:    {r.type('myset')}")      # set
print(f"myhash type:   {r.type('myhash')}")     # hash

# RENAME — Rename a key
# Redis CLI: RENAME mystring greeting
r.rename('mystring', 'greeting')
print(f"\nAfter rename: {r.get('greeting')}")
print(f"Old key exists? {r.exists('mystring')}")

In [ ]:
# OBJECT ENCODING — How Redis stores the key internally
r.set('small_number', '42')
r.set('big_string', 'a' * 100)

print(f"small_number encoding: {r.object('encoding', 'small_number')}")  # int
print(f"big_string encoding:   {r.object('encoding', 'big_string')}")    # embstr or raw

# DBSIZE — Total number of keys
print(f"\nTotal keys in DB: {r.dbsize()}")

---
## 11. Real-World: Session Management

In [ ]:
import uuid

SESSION_TIMEOUT = 10  # 10 seconds for demo (normally 1800 = 30 min)

def login(username):
    """Create a session for a user."""
    session_id = str(uuid.uuid4())
    key = f'session:{session_id}'
    r.hset(key, mapping={'username': username, 'login_time': str(time.time())})
    r.expire(key, SESSION_TIMEOUT)
    print(f"  Logged in {username}, session: {session_id[:8]}...")
    return session_id

def check_session(session_id):
    """Check if session is valid. Refresh timeout if so."""
    key = f'session:{session_id}'
    data = r.hgetall(key)
    if data:
        r.expire(key, SESSION_TIMEOUT)  # Refresh!
        return data
    return None

def logout(session_id):
    """Destroy session."""
    r.delete(f'session:{session_id}')
    print(f"  Logged out session {session_id[:8]}...")

# Demo
sid = login('sujit')

# Check immediately — should work
data = check_session(sid)
print(f"  Session valid: {data}")
print(f"  TTL: {r.ttl(f'session:{sid}')}s")

# Wait and check again
print("  Waiting 3 seconds...")
time.sleep(3)
data = check_session(sid)  # This refreshes the timeout!
print(f"  Session still valid: {data is not None}")
print(f"  TTL refreshed to: {r.ttl(f'session:{sid}')}s")

---
## 12. Real-World: OTP (One-Time Password)

In [ ]:
import random

def generate_otp(user_id, ttl=300):  # 5 minutes default
    """Generate a 6-digit OTP that auto-expires."""
    otp = str(random.randint(100000, 999999))
    key = f'otp:{user_id}'
    r.set(key, otp, ex=ttl)
    return otp

def verify_otp(user_id, code):
    """Verify OTP — can only be used once (GETDEL)."""
    key = f'otp:{user_id}'
    stored = r.getdel(key)  # Get and delete in one atomic op
    if stored is None:
        return False, "OTP expired or not found"
    if stored == code:
        return True, "OTP verified!"
    else:
        return False, "Invalid OTP"

# Demo
otp = generate_otp('sujit', ttl=10)  # 10 sec for demo
print(f"Generated OTP: {otp}")
print(f"TTL: {r.ttl('otp:sujit')} seconds")

# Verify with correct code
success, msg = verify_otp('sujit', otp)
print(f"\nFirst verify: {msg} (success={success})")

# Try again — should fail (GETDEL already deleted it)
success, msg = verify_otp('sujit', otp)
print(f"Second verify: {msg} (success={success})")
print("OTP can only be used once!")

---
## 13. Real-World: Cache with TTL

In [ ]:
import json

def expensive_db_query(query):
    """Simulate a slow database query."""
    time.sleep(1)  # Simulates 1 second of work
    return {'result': f'data for {query}', 'rows': 42}

def cached_query(query, ttl=30):
    """Query with caching. Returns (result, from_cache)."""
    cache_key = f'cache:query:{query}'
    
    # Try cache first
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # Cache hit!
    
    # Cache miss — run the expensive query
    result = expensive_db_query(query)
    r.set(cache_key, json.dumps(result), ex=ttl)
    return result, False

# First call — cache miss (slow)
start = time.time()
result, from_cache = cached_query('SELECT * FROM users')
print(f"Call 1: {time.time()-start:.2f}s, cache={'HIT' if from_cache else 'MISS'}")

# Second call — cache hit (fast!)
start = time.time()
result, from_cache = cached_query('SELECT * FROM users')
print(f"Call 2: {time.time()-start:.4f}s, cache={'HIT' if from_cache else 'MISS'}")

print(f"\nResult: {result}")
print(f"Cache TTL: {r.ttl('cache:query:SELECT * FROM users')}s")

---
## 14. Memory Management Tips

In [ ]:
# Check memory usage of a specific key
r.set('small', 'hi')
r.set('large', 'x' * 10000)

print(f"'small' memory: {r.memory_usage('small')} bytes")
print(f"'large' memory: {r.memory_usage('large')} bytes")

# Overall database stats
info = r.info('memory')
print(f"\nTotal used memory: {info['used_memory_human']}")
print(f"Peak memory: {info['used_memory_peak_human']}")
print(f"Total keys: {r.dbsize()}")

# Stats about cache hits/misses
stats = r.info('stats')
hits = stats.get('keyspace_hits', 0)
misses = stats.get('keyspace_misses', 0)
total = hits + misses
ratio = (hits / total * 100) if total > 0 else 0
print(f"\nCache hits: {hits}, misses: {misses}")
print(f"Hit ratio: {ratio:.1f}%")

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

```
SET key val EX seconds    → Set with expiry
EXPIRE key seconds        → Add expiry to existing key
PEXPIRE key ms            → Expiry in milliseconds
EXPIREAT key timestamp    → Expire at specific time
TTL key                   → Check remaining seconds (-1=permanent, -2=gone)
PTTL key                  → Check remaining milliseconds
PERSIST key               → Remove expiry
KEYS pattern              → Find keys (DON'T use in production!)
SCAN 0 MATCH pattern      → Safe key iteration
TYPE key                  → Get data type
RENAME key newkey         → Rename a key
MEMORY USAGE key          → Check memory of a key
DBSIZE                    → Count all keys
INFO memory/stats         → Server statistics
```

### Golden Rules
1. **Always set TTL on cache keys** — prevents memory leaks
2. **Use SCAN, never KEYS** in production
3. **Use consistent key naming** with colon separators
4. **GETDEL for one-time tokens** — read and destroy atomically

---
## Exercises

1. **Expiry Explorer:** Create 5 keys with TTLs of 5, 10, 15, 20, and 25 seconds. Write a loop that checks all 5 keys every 5 seconds and prints which are still alive.

2. **Token Service:** Write functions `create_token(user_id)` and `validate_token(user_id, token)`. Tokens should expire after 60 seconds and be single-use.

3. **Key Namespace Counter:** Write a function that counts how many keys exist for each prefix (e.g., `user:*` → 50, `cache:*` → 120). Use SCAN, not KEYS.

4. **Auto-Expiring Cache:** Build a cache decorator that you can add to any function: `@redis_cache(ttl=30)`. It should cache the function's return value in Redis with the given TTL.

5. **Session Cleanup Report:** Create 10 sessions with random TTLs. Write a function that reports: total sessions, expired sessions, sessions expiring in < 5 seconds.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 07 — Pub/Sub](./07_Pub_Sub.ipynb)** — Real-time messaging, chat rooms, and event systems!